# 06 — Daily Exposure + Synthetic Health

Aggregate hourly → daily và tạo health outcome mô phỏng để test GAM. Đây không phải dữ liệu bệnh viện thật.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shutil

# ------------------------------------------------------------
# Robust project root finder
# Notebook lives under: Project/notebooks/son_nb/
# ------------------------------------------------------------
def find_project_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / "data" / "raw").exists() and (p / "notebooks").exists():
            return p
    raise FileNotFoundError(
        "Không tìm thấy Project root. Hãy chạy notebook bên trong project."
    )

PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MAIN_OUTPUT = PROJECT_ROOT / "outputs"
SON_DIR = PROJECT_ROOT / "notebooks" / "son_nb"
SON_OUTPUT = SON_DIR / "output"

WORK_DATA = SON_OUTPUT / "data"
WORK_FIG = SON_OUTPUT / "figures"
WORK_TABLE = SON_OUTPUT / "tables"
WORK_PRED = SON_OUTPUT / "predictions"
WORK_MODEL = SON_OUTPUT / "models"

for p in [WORK_DATA, WORK_FIG, WORK_TABLE, WORK_PRED, WORK_MODEL]:
    p.mkdir(parents=True, exist_ok=True)

BEIJING_RAW = RAW_DIR / "beijing" / "PRSA_data_2010.1.1-2014.12.31.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BEIJING_RAW:", BEIJING_RAW)
print("Working output:", SON_OUTPUT)

In [ ]:
df=pd.read_csv(WORK_DATA/'beijing_hourly_clean_working.csv',parse_dates=['datetime']).sort_values('datetime').set_index('datetime')
daily=df.resample('D').agg({'pm2.5':'mean','TEMP':'mean','DEWP':'mean','PRES':'mean','Iws':'mean','Ir':'sum'})
daily['pm25_valid_hours']=df['pm2.5'].resample('D').count(); daily['pm25_valid_day']=daily.pm25_valid_hours>=20; daily.loc[~daily.pm25_valid_day,'pm2.5']=np.nan
daily=daily.reset_index(); daily['dayofweek']=daily.datetime.dt.dayofweek; daily['dayofyear']=daily.datetime.dt.dayofyear; daily['time_index']=np.arange(len(daily))

### Synthetic health design
Paper Core chỉ cung cấp logic phương pháp và RR tham khảo. Ở đây dùng tham số Core-inspired để kiểm tra khả năng recovery của GAM; **không gọi đây là RR của Beijing**.

In [ ]:
BETA_LAG0 = np.log(1.0195) / 10
BETA_LAG1 = np.log(1.0131) / 10

s = daily.copy()
s["pm25_lag0"] = s["pm2.5"]
s["pm25_lag1"] = s["pm2.5"].shift(1)

base = (
    np.log(30.0)
    + 0.12 * np.sin(2 * np.pi * s["dayofyear"] / 365.25)
    + np.where(s["dayofweek"] >= 5, -0.05, 0.0)
    + 0.01 * (s["TEMP"] - s["TEMP"].median())
)

rng = np.random.default_rng(42)

s["resp_cases_null"] = rng.poisson(
    np.exp(base).clip(lower=0.1)
)

s["resp_cases_lag0"] = rng.poisson(
    np.exp(
        base
        + BETA_LAG0 * s["pm25_lag0"].fillna(s["pm2.5"].median())
    ).clip(lower=0.1)
)

s["resp_cases_lag01"] = rng.poisson(
    np.exp(
        base
        + BETA_LAG0 * s["pm25_lag0"].fillna(s["pm2.5"].median())
        + BETA_LAG1 * s["pm25_lag1"].fillna(s["pm2.5"].median())
    ).clip(lower=0.1)
)

health = s[
    [
        "datetime", "pm2.5", "pm25_valid_hours",
        "TEMP", "DEWP", "PRES", "Iws", "Ir",
        "dayofweek", "dayofyear", "time_index",
        "resp_cases_lag01"
    ]
].rename(columns={"resp_cases_lag01": "resp_cases"})

health.to_csv(
    WORK_DATA / "beijing_daily_health_simulated_working.csv",
    index=False
)

display(health.head())